In [1]:
!pip install -q ultralytics supervision

In [1]:
import os, csv, cv2, json
from typing import Optional, Iterable, Dict, Tuple, List

# -------- paths --------
VIDEO_PATH   = "/content/vt_road_2.MOV"
WEIGHTS_PT   = "/content/video_analytics_training_models_garbage_overflowing_2_runs_detect_train_weights_best (1).pt"
OUTPUT_PATH  = "/content/vt_road_2_garbage_output.mp4"

# per-detection log (ONLY when a detection is first seen = first occurrence of a track_id)
CSV_PATH            = "/content/inference_first_occurrence.csv"
# per-frame summary of NEW uniques that appeared in THIS frame only
FRAME_SUMMARY_CSV   = "/content/frame_new_uniques_summary_4.csv"

# Optional: per-frame GPS CSV: columns => frame_idx,lat,lon
FRAME_GPS_CSV: Optional[str] = None
# -------- class filter --------
# Keep only these classes by NAME; set [] or None to keep ALL.
ALLOWED_CLASSES: Optional[Iterable[str]] = [
    "garbage"
]

# -------- inference thresholds --------
CONF_THRESH = 0.25
IOU_THRESH  = 0.3

# -------- drawing toggles --------
DRAW_BOXES  = True
DRAW_LABELS = True
DRAW_MASKS  = True   # if segmentation masks exist

print("VIDEO_PATH:", VIDEO_PATH)
print("WEIGHTS_PT:", WEIGHTS_PT)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("CSV_PATH:", CSV_PATH)
print("FRAME_SUMMARY_CSV:", FRAME_SUMMARY_CSV)
print("FRAME_GPS_CSV:", FRAME_GPS_CSV)
print("ALLOWED_CLASSES:", ALLOWED_CLASSES if ALLOWED_CLASSES else "ALL")
print("CONF/IOU:", CONF_THRESH, IOU_THRESH)


VIDEO_PATH: /content/vt_road_2.MOV
WEIGHTS_PT: /content/video_analytics_training_models_garbage_overflowing_2_runs_detect_train_weights_best (1).pt
OUTPUT_PATH: /content/vt_road_2_garbage_output.mp4
CSV_PATH: /content/inference_first_occurrence.csv
FRAME_SUMMARY_CSV: /content/frame_new_uniques_summary_4.csv
FRAME_GPS_CSV: None
ALLOWED_CLASSES: ['garbage']
CONF/IOU: 0.25 0.3


In [2]:
def ensure_detection_csv_header(path: str):
    new_file = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new_file:
            # track_id makes uniqueness explicit; confidence is the model's score (not CONF_THRESH)
            w.writerow(["track_id", "confidence", "lat", "lon", "video_timestamp", "class_name", "frame_idx"])

def append_detection_row(path: str, track_id: int, confidence: float, lat, lon,
                         ts_str: str, class_name: str, frame_idx: int):
    with open(path, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([
            int(track_id) if track_id is not None else "",
            round(float(confidence), 6) if confidence is not None else "",
            lat if lat is not None else "",
            lon if lon is not None else "",
            ts_str,
            class_name,
            frame_idx
        ])

def ensure_frame_summary_header(path: str):
    new_file = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new_file:
            # counts for THIS frame only (new uniques first seen now)
            w.writerow(["video_timestamp", "frame_idx", "new_unique_total", "new_unique_class_counts_json"])

def append_frame_summary_row(path: str, ts_str: str, frame_idx: int,
                             new_unique_total: int, counts_by_class: Dict[str, int]):
    with open(path, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([ts_str, frame_idx, new_unique_total, json.dumps(counts_by_class, ensure_ascii=False)])

def hms_from_msec(ms: float) -> str:
    s, ms = divmod(ms, 1000.0); s = int(s)
    h, s = divmod(s, 3600); m, s = divmod(s, 60)
    return f"{h:02d}:{m:02d}:{s:02d}.{int(ms):03d}"

def load_frame_gps_map(csv_path: Optional[str]) -> Dict[int, Tuple[float, float]]:
    mapping: Dict[int, Tuple[float, float]] = {}
    if not csv_path or not os.path.exists(csv_path):
        return mapping
    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                fi  = int(str(row.get("frame_idx","")).strip())
                lat = float(str(row.get("lat","")).strip())
                lon = float(str(row.get("lon","")).strip())
                mapping[fi] = (lat, lon)
            except Exception:
                pass
    return mapping


In [3]:
from ultralytics import YOLO

# supervision (with version info, shims)
try:
    import supervision as sv
    try:
        print("supervision version:", sv.__version__)
    except Exception:
        pass
except Exception:
    sv = None
    print("supervision not available; drawing will be limited.")

model = YOLO(WEIGHTS_PT)

allowed_set = None if not ALLOWED_CLASSES else set([x.strip().lower() for x in ALLOWED_CLASSES])

print("Model classes:", model.names)

if allowed_set:
    known = {str(v).lower() for v in model.names.values()}
    unknown = [n for n in allowed_set if n not in known]
    if unknown:
        print("⚠️ Not in model.names:", unknown)

# ---- open video ----
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open {VIDEO_PATH}")

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_in = cap.get(cv2.CAP_PROP_FPS)

# sanitize FPS: some sources report 0 or weird fractions
fps = int(round(fps_in)) if fps_in and fps_in > 1e-3 else 25
fps = max(1, min(fps, 120))

print(f"Input video: {width}x{height} @ {fps_in:.3f} fps (writing @ {fps} fps)")

# H.264/yuv420p on Windows often needs even dimensions
out_w = width  - (width  % 2)
out_h = height - (height % 2)
if (out_w, out_h) != (width, height):
    print(f"Resizing frames to even dims for encoding: {width}x{height} -> {out_w}x{out_h}")

# --- robust writer with fallbacks ---
def make_writer(base_out_path: str, size, fps_val) -> tuple:
    """
    Try several codecs; return (writer, out_path, fourcc_used).
    On Windows, XVID/MJPG in .avi are the most universally playable.
    """
    base, _ext = os.path.splitext(base_out_path)
    candidates = [
        ("XVID", ".avi"),   # ✅ very compatible on Windows (AVI)
        ("MJPG", ".avi"),   # ✅ also very compatible (large files)
        ("mp4v", ".mp4"),   # ⚠️ MPEG-4 Part 2 in MP4 (often not playable on WMP)
        ("avc1", ".mp4"),   # H.264 if your OpenCV+FFmpeg supports it
        ("H264", ".mp4"),
    ]
    for fourcc, ext in candidates:
        out_path_try = base + ext
        w = cv2.VideoWriter(out_path_try, cv2.VideoWriter_fourcc(*fourcc), fps_val, size)
        if w.isOpened():
            print(f"✅ Using codec {fourcc} -> {out_path_try}")
            return w, out_path_try, fourcc
        else:
            print(f"⚠️ Failed to open writer with {fourcc} {ext}")
    raise RuntimeError("Could not open any VideoWriter (no working codec).")

out, OUTPUT_PATH_FINAL, OUT_CODEC = make_writer(OUTPUT_PATH, (out_w, out_h), fps)

# ---- annotators (version-safe) ----
box_annot = mask_annot = label_anno = None

def safe_box_annotate(img, detections, labels):
    if box_annot is None:
        return img
    try:
        return box_annot.annotate(img, detections, labels=labels)
    except TypeError:
        pass
    try:
        return box_annot.annotate(img, detections)
    except TypeError:
        pass
    try:
        return box_annot.annotate(scene=img, detections=detections, labels=labels)
    except TypeError:
        pass
    try:
        return box_annot.annotate(scene=img, detections=detections)
    except TypeError:
        pass
    return img

def safe_label_annotate(img, detections, labels):
    if label_anno is None:
        return img
    try:
        return label_anno.annotate(img, detections, labels=labels)
    except TypeError:
        pass
    try:
        return label_anno.annotate(scene=img, detections=detections, labels=labels)
    except TypeError:
        pass
    try:
        return label_anno.annotate(img, detections, texts=labels)
    except TypeError:
        pass
    try:
        return label_anno.annotate(scene=img, detections=detections, texts=labels)
    except TypeError:
        pass
    return img

def draw_labels_fallback(img, detections, labels):
    if detections is None or len(detections) == 0:
        return img
    for (xyxy, txt) in zip(detections.xyxy, labels):
        x1, y1, x2, y2 = map(int, xyxy)
        pad = 3
        (tw, th), baseline = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        rx1, ry1 = x1, max(0, y1 - th - baseline - 2*pad)
        rx2, ry2 = x1 + tw + 2*pad, y1
        cv2.rectangle(img, (rx1, ry1), (rx2, ry2), (0, 0, 0), -1)
        cv2.putText(img, txt, (x1 + pad, y1 - baseline - pad),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1, cv2.LINE_AA)
    return img

if sv is not None:
    if hasattr(sv, "BoxAnnotator"):
        box_annot = sv.BoxAnnotator(thickness=2)
    elif hasattr(sv, "BoundingBoxAnnotator"):
        box_annot = sv.BoundingBoxAnnotator(thickness=2)
    if hasattr(sv, "LabelAnnotator"):
        label_anno = sv.LabelAnnotator(text_scale=0.5, text_thickness=1, text_padding=4)
    if hasattr(sv, "MaskAnnotator"):
        mask_annot = sv.MaskAnnotator(opacity=0.4)

# ---- tracker: prefer ByteTrack; provide IoU fallback ----
tracker = None
iou_fallback = True

if sv is not None:
    try:
        tracker = sv.ByteTrack()
        iou_fallback = False
        print("Using ByteTrack from supervision.")
    except Exception:
        try:
            from supervision.tracker.byte_track import ByteTrack
            tracker = ByteTrack()
            iou_fallback = False
            print("Using ByteTrack (legacy import).")
        except Exception:
            print("ByteTrack not available; will use IoU fallback.")

# IoU fallback state
tracks = {}           # id -> dict(box, cls, last_frame)
next_tid = 1
IOU_MATCH_THRESH = 0.5
MAX_AGE_FRAMES  = int(fps * 1.0)  # 1s tolerance  (uses the sanitized fps above)

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    iw = max(0, inter_x2 - inter_x1)
    ih = max(0, inter_y2 - inter_y1)
    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

# GPS + CSV headers + seen set
frame_gps_map = load_frame_gps_map(FRAME_GPS_CSV)
ensure_detection_csv_header(CSV_PATH)
ensure_frame_summary_header(FRAME_SUMMARY_CSV)
seen_global_ids = set()     # global uniqueness across the entire video (track-based)

print("Setup complete.  Output will be written to:", OUTPUT_PATH_FINAL)

supervision version: 0.26.1
Model classes: {0: 'garbage', 1: 'rubbish'}
Input video: 3840x2160 @ 59.975 fps (writing @ 60 fps)
✅ Using codec XVID -> /content/vt_road_2_garbage_output.avi
Using ByteTrack from supervision.
Setup complete.  Output will be written to: /content/vt_road_2_garbage_output.avi


In [4]:
from typing import List
import numpy as np

frame_idx = -1
while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame_idx += 1

    # Predict
    results = model.predict(source=frame, conf=CONF_THRESH, iou=IOU_THRESH, verbose=False, stream=False)
    if not results:
        out.write(frame)
        continue
    res = results[0]

    # Timecode + GPS
    ts_str = hms_from_msec(cap.get(cv2.CAP_PROP_POS_MSEC))
    lat = lon = None
    if frame_idx in frame_gps_map:
        lat, lon = frame_gps_map[frame_idx]

    # Convert to supervision Detections
    detections = None
    if sv is not None and hasattr(sv, "Detections"):
        detections = sv.Detections.from_ultralytics(res)

        # Filter by class names if specified
        if allowed_set and len(detections) > 0:
            keep_idx = []
            for i, cid in enumerate(detections.class_id):
                name = str(model.names.get(int(cid), str(cid))).strip().lower()
                if name in allowed_set:
                    keep_idx.append(i)
            detections = detections[keep_idx] if keep_idx else sv.Detections.empty()

    # We also need confidences and xyxy for logging/tracking
    # Use res.boxes directly to get confidence per detection
    # Build aligned lists for filtered detections
    xyxy_list = []
    cls_list  = []
    conf_list = []

    if hasattr(res, "boxes") and res.boxes is not None and len(res.boxes) > 0:
        for i in range(len(res.boxes)):
            cid   = int(res.boxes.cls[i].item())
            cname = str(model.names.get(cid, str(cid))).strip()
            if allowed_set and cname.lower() not in allowed_set:
                continue
            xyxy = res.boxes.xyxy[i].tolist()
            conf = float(res.boxes.conf[i].item()) if hasattr(res.boxes, "conf") else None
            xyxy_list.append(xyxy)
            cls_list.append(cid)
            conf_list.append(conf)

    # ----- tracking to assign persistent IDs -----
    track_ids = []
    if sv is not None and detections is not None and not iou_fallback:
        # ByteTrack path
        tracked = tracker.update_with_detections(detections)
        # tracked.tracker_id aligns with tracked.xyxy / detections order
        # But we need to align with our xyxy_list (which was filtered identically above)
        # Build a quick map by (x1,y1,x2,y2,cls) rounding to avoid float issues
        def key_of(x, c): return tuple([int(round(v)) for v in x] + [int(c)])
        det_index = { key_of(detections.xyxy[i], detections.class_id[i]) : i for i in range(len(detections)) }
        tr_index  = { key_of(tracked.xyxy[i],   tracked.class_id[i])   : i for i in range(len(tracked)) }
        for x, c in zip(xyxy_list, cls_list):
            k = key_of(x, c)
            if k in tr_index:
                tid = tracked.tracker_id[tr_index[k]]
            else:
                tid = None
            track_ids.append(int(tid) if tid is not None else None)
    else:
        # IoU fallback: naive association to maintain IDs
        # expire old tracks
        to_del = []
        for tid, st in tracks.items():
            if frame_idx - st["last_frame"] > MAX_AGE_FRAMES:
                to_del.append(tid)
        for tid in to_del:
            del tracks[tid]

        # assign current detections
        for x, c in zip(xyxy_list, cls_list):
            best_iou, best_tid = 0.0, None
            for tid, st in tracks.items():
                if st["cls"] != c:
                    continue
                iou = iou_xyxy(x, st["box"])
                if iou > best_iou:
                    best_iou, best_tid = iou, tid
            if best_iou >= IOU_MATCH_THRESH:
                # reuse
                tracks[best_tid]["box"] = x
                tracks[best_tid]["last_frame"] = frame_idx
                track_ids.append(best_tid)
            else:
                # new id
                global next_tid
                tid = next_tid
                next_tid += 1
                tracks[tid] = {"box": x, "cls": c, "last_frame": frame_idx}
                track_ids.append(tid)

    # ----- log ONLY first occurrence of each track_id -----
    new_ids_this_frame = []
    counts_by_class = {}

    for x, c, conf, tid in zip(xyxy_list, cls_list, conf_list, track_ids):
        cname = str(model.names.get(int(c), str(c))).strip()
        # If we don't have a tracker_id, treat all as new this frame (but that's not desired).
        # With ByteTrack or IoU fallback, tid exists. Only log first time we see this tid.
        if tid is None:
            continue
        if tid not in seen_global_ids:
            seen_global_ids.add(tid)
            new_ids_this_frame.append(tid)
            counts_by_class[cname] = counts_by_class.get(cname, 0) + 1
            # LOG the first occurrence with the model's CONFIDENCE (per detection)
            append_detection_row(CSV_PATH, tid, conf, lat, lon, ts_str, cname, frame_idx)

    append_frame_summary_row(FRAME_SUMMARY_CSV, ts_str, frame_idx, len(new_ids_this_frame), counts_by_class)

    # ----- draw -----
    drawn = frame.copy()
    if sv is not None and hasattr(sv, "Detections"):
        # For drawing, recreate detections with the same filtering as xyxy_list
        if len(xyxy_list) > 0:
            # Build a fresh Detections object from lists so order matches track_ids/conf
            dets = sv.Detections(
                xyxy=np.array(xyxy_list, dtype=float),
                class_id=np.array(cls_list, dtype=int),
                confidence=np.array(conf_list, dtype=float)
            )
            labels = []
            for i, c in enumerate(cls_list):
                cname = str(model.names.get(int(c), str(c)))
                conf  = conf_list[i]
                tid   = track_ids[i]
                if tid is not None:
                    labels.append(f"{cname} #{tid} {conf:.2f}")
                else:
                    labels.append(f"{cname} {conf:.2f}")

            if DRAW_BOXES and box_annot is not None:
                drawn = safe_box_annotate(drawn, dets, labels)
            if DRAW_MASKS and mask_annot is not None and getattr(res, "masks", None) is not None:
                try:
                    drawn = mask_annot.annotate(drawn, dets)
                except TypeError:
                    try:
                        drawn = mask_annot.annotate(scene=drawn, detections=dets)
                    except TypeError:
                        pass
            if DRAW_LABELS:
                drawn = safe_label_annotate(drawn, dets, labels) if label_anno is not None else draw_labels_fallback(drawn, dets, labels)

    # ensure even-size frame for encoder
    if drawn.shape[1] != out_w or drawn.shape[0] != out_h:
        frame_to_write = cv2.resize(drawn, (out_w, out_h), interpolation=cv2.INTER_AREA)
    else:
        frame_to_write = drawn

    out.write(frame_to_write)

print("Loop finished.")


Loop finished.
